# MuseTalk 1.5 — T4 worker

Clean Google Colab T4 worker. It creates a fresh Python 3.10 environment, installs the CUDA stack without using the failing `pip upgrade` step, downloads the required MuseTalk weights, runs inference, and stops on the real error.

In [ ]:
# 1) Create a clean Python 3.10 GPU environment
import os, sys, shutil, subprocess
from pathlib import Path

print("Checking Colab GPU...")
subprocess.run(["nvidia-smi"], check=True)

MT = Path("/content/MuseTalk")
VENV = Path("/content/musetalk310")

if not MT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/TMElyralab/MuseTalk.git", str(MT)], check=True)

# Recreate an incomplete/broken venv instead of reusing it.
need_recreate = not (VENV / "bin/python").exists()
if not need_recreate:
    probe = subprocess.run([str(VENV / "bin/python"), "-c", "import torch"], capture_output=True)
    need_recreate = probe.returncode != 0

if need_recreate and VENV.exists():
    print("Removing incomplete MuseTalk Python environment...")
    shutil.rmtree(VENV)

if not VENV.exists():
    subprocess.run(["uv", "venv", "--python", "3.10", "--seed", str(VENV)], check=True)

PY = str(VENV / "bin/python")
UV = ["uv", "pip", "install", "--python", PY]

# IMPORTANT: do not run `python -m pip install --upgrade pip...`.
# That is the step that has been failing in the current notebook.
subprocess.run(UV + ["pip==24.0", "setuptools==69.5.1", "wheel==0.43.0"], check=True)

# Known-good T4/CUDA 11.8 stack used by MuseTalk's Python 3.10 setup.
subprocess.run(UV + ["torch==2.0.1", "torchvision==0.15.2", "torchaudio==2.0.2", "--index-url", "https://download.pytorch.org/whl/cu118"], check=True)

probe = subprocess.run([PY, "-c", 'import torch; print(torch.__version__); print(torch.version.cuda); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'], text=True, capture_output=True, check=True)
print(probe.stdout)

if probe.stdout.strip().splitlines()[-2] != "True":
    raise RuntimeError("STOP: MuseTalk Python 3.10 cannot see the T4 CUDA GPU.")

print("T4 CUDA environment is ready.")


In [ ]:
# 2) Install dependencies and download all required MuseTalk weights
import subprocess, os
from pathlib import Path

PY = "/content/musetalk310/bin/python"
MT = "/content/MuseTalk"
MODELS = Path(MT) / "models"
MODELS.mkdir(parents=True, exist_ok=True)
UV = ["uv", "pip", "install", "--python", PY]

# Install requirements with uv; no pip self-upgrade and no hidden fallback.
subprocess.run(UV + ["-r", MT + "/requirements.txt"], check=True)
subprocess.run(UV + ["huggingface_hub==0.30.2", "gdown", "openmim"], check=True)

# Install OpenMMLab packages only after PyTorch is installed.
MIM = "/content/musetalk310/bin/mim"
for pkg in ["mmengine", "mmcv==2.0.1", "mmdet==3.1.0", "mmpose==1.1.0"]:
    p = subprocess.run([MIM, "install", pkg], text=True, capture_output=True)
    print(p.stdout)
    if p.returncode:
        print(p.stderr)
        raise RuntimeError(f"MMLab install failed: {pkg}")

HF = "/content/musetalk310/bin/hf"
if not Path(HF).exists():
    raise RuntimeError("Hugging Face CLI was not installed in the Python 3.10 environment.")

def hf_download(repo, local_dir, *files):
    Path(local_dir).mkdir(parents=True, exist_ok=True)
    cmd = [HF, "download", repo, "--local-dir", str(local_dir)]
    for f in files:
        cmd += ["--include", f]
    subprocess.run(cmd, check=True)

hf_download("TMElyralab/MuseTalk", MODELS, "musetalkV15/unet.pth", "musetalkV15/musetalk.json")
hf_download("stabilityai/sd-vae-ft-mse", MODELS / "sd-vae", "config.json", "diffusion_pytorch_model.bin")
hf_download("openai/whisper-tiny", MODELS / "whisper", "config.json", "preprocessor_config.json", "pytorch_model.bin")
hf_download("yzd-v/DWPose", MODELS / "dwpose", "dw-ll_ucoco_384.pth")
hf_download("ManyOtherFunctions/face-parse-bisent", MODELS / "face-parse-bisent", "79999_iter.pth", "resnet18-5c106cde.pth")

# Force the VAE loader to use the downloaded PyTorch .bin weights.
vae_py = Path(MT) / "musetalk/models/vae.py"
txt = vae_py.read_text()
txt = txt.replace("AutoencoderKL.from_pretrained(self.model_path)", "AutoencoderKL.from_pretrained(self.model_path, use_safetensors=False)")
vae_py.write_text(txt)

required = [
    MODELS / "musetalkV15/unet.pth",
    MODELS / "musetalkV15/musetalk.json",
    MODELS / "sd-vae/config.json",
    MODELS / "sd-vae/diffusion_pytorch_model.bin",
    MODELS / "whisper/config.json",
    MODELS / "whisper/preprocessor_config.json",
    MODELS / "whisper/pytorch_model.bin",
    MODELS / "dwpose/dw-ll_ucoco_384.pth",
    MODELS / "face-parse-bisent/79999_iter.pth",
    MODELS / "face-parse-bisent/resnet18-5c106cde.pth",
]
for f in required:
    if not f.exists() or f.stat().st_size < 1000:
        raise RuntimeError(f"Missing/incomplete model file: {f}")

print("All MuseTalk 1.5 model files are ready.")


In [ ]:
# 3) Upload approved singer image/video + ACE-Step audio and run MuseTalk
from google.colab import files
from pathlib import Path
import subprocess, os

MT = Path("/content/MuseTalk")
OUT = Path("/content/musetalk_output")
OUT.mkdir(parents=True, exist_ok=True)

print("Upload the APPROVED singer image/video:")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No singer image/video was uploaded.")
avatar = next(iter(uploaded))

print("Upload the successful ACE-Step bhajan MP3/WAV:")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No audio was uploaded.")
audio = next(iter(uploaded))

avatar_src = OUT / "avatar_source.png"
audio_wav = OUT / "audio.wav"
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", avatar, "-frames:v", "1", "-vf", "scale=512:-2", "-pix_fmt", "rgb24", str(avatar_src)], check=True)
subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", audio, "-ar", "16000", "-ac", "1", str(audio_wav)], check=True)

cfg = MT / "configs/inference/test.yaml"
cfg.write_text(f'bhajan_test:\n  video_path: "{avatar_src}"\n  audio_path: "{audio_wav}"\n  result_name: "bhajan_lipsync.mp4"\n')

# Make MuseTalk re-raise its real exception instead of silently continuing.
inf_py = MT / "scripts/inference.py"
itxt = inf_py.read_text()
old = '        except Exception as e:\n            print("Error occurred during processing:", e)\n'
new = '        except Exception as e:\n            print("Error occurred during processing:", e)\n            raise\n'
if old in itxt:
    inf_py.write_text(itxt.replace(old, new, 1))

os.chdir(MT)
env = os.environ.copy()
env["MPLBACKEND"] = "Agg"
env["PYTHONPATH"] = str(MT) + os.pathsep + env.get("PYTHONPATH", "")
env["CUDA_VISIBLE_DEVICES"] = "0"

pre = subprocess.run(["/content/musetalk310/bin/python", "-c", "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"], text=True, capture_output=True)
if pre.returncode != 0:
    raise RuntimeError("STOP: T4 is not visible to MuseTalk.\n" + pre.stderr)
print("Using GPU:", pre.stdout.strip())

result_dir = OUT / "result"
result_dir.mkdir(parents=True, exist_ok=True)
cmd = [
    "/content/musetalk310/bin/python", "-m", "scripts.inference",
    "--inference_config", "configs/inference/test.yaml",
    "--result_dir", str(result_dir),
    "--unet_model_path", "models/musetalkV15/unet.pth",
    "--unet_config", "models/musetalkV15/musetalk.json",
    "--whisper_dir", "models/whisper",
    "--version", "v15",
    "--fps", "25",
    "--batch_size", "2",
    "--use_float16",
    "--parsing_mode", "jaw",
]

print("Starting MuseTalk 1.5 on T4...")
r = subprocess.run(cmd, env=env, text=True, capture_output=True)
print(r.stdout)
if r.stderr:
    print(r.stderr)

if r.returncode != 0:
    raise RuntimeError(f"MuseTalk inference failed with exit code {r.returncode}. The complete STDERR/STDOUT is printed immediately above.")

candidates = list(result_dir.rglob("*.mp4"))
if not candidates:
    raise RuntimeError("MuseTalk exited successfully but produced no MP4.")

candidate = max(candidates, key=lambda p: p.stat().st_size)
if candidate.stat().st_size < 100000:
    raise RuntimeError(f"Output MP4 is suspiciously small: {candidate}")

print("SUCCESS:", candidate)
print("Size:", round(candidate.stat().st_size / 1024 / 1024, 2), "MB")
files.download(str(candidate))
